In [1]:
import json

In [2]:
with open(r'acts.json', 'r') as file:
    all_acts_links = json.load(file)

In [3]:
out_folder = "./acts_pdf"

In [4]:
from bs4 import BeautifulSoup
import requests
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from urllib.parse import urljoin
import cloudscraper

# Headers to avoid being blocked
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Connection": "keep-alive",
    "DNT": "1",
    "Upgrade-Insecure-Requests": "1",
}

scraper = cloudscraper.create_scraper()
scraper.headers.update(headers)

# Create output directory if it doesn't exist
os.makedirs(out_folder, exist_ok=True)

prefix = "https://www.indiacode.nic.in"

# Number of retries for each download
MAX_RETRIES = 3
RETRY_DELAY = 2  # seconds

def download_pdf_from_url(url, index, total_urls):
    """
    Process a single URL to find and download ALL PDFs with retry logic
    """
    results = []
    try:
        print(f"Processing link {index+1}/{total_urls}: {url}")
        response = scraper.get(url, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, "html.parser")
        links = soup.find_all("a", href=True)

        # Find all PDF links except userGuide.pdf
        pdf_links = [a['href'] for a in links if ".pdf" in a['href'] and "userGuide.pdf" not in a['href']]
        if not pdf_links:
            print(f"✗ No PDF found at: {url}")
            return [{"status": "no_pdf_found", "url": url}]

        for href in pdf_links:
            pdf_url = urljoin(prefix, href)
            filename = href.split('/')[-1]
            success = False

            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    print(f"Attempt {attempt}: Downloading PDF: {filename}")
                    file_resp = scraper.get(pdf_url, timeout=60)
                    file_resp.raise_for_status()

                    filepath = os.path.join(out_folder, filename)
                    with open(filepath, "wb") as f:
                        f.write(file_resp.content)

                    print(f"✓ Successfully downloaded: {filename}")
                    results.append({"status": "success", "url": url, "filename": filename})
                    success = True
                    break

                except requests.exceptions.RequestException as e:
                    error_msg = f"Attempt {attempt} failed for {pdf_url}: {e}"
                    print(f"✗ {error_msg}")
                    if attempt < MAX_RETRIES:
                        print(f"Retrying in {RETRY_DELAY} seconds...")
                        time.sleep(RETRY_DELAY)
                    else:
                        results.append({"status": "pdf_download_error", "url": url, "error": error_msg})
                except IOError as e:
                    error_msg = f"Failed to save PDF {filename} on attempt {attempt}: {e}"
                    print(f"✗ {error_msg}")
                    results.append({"status": "file_save_error", "url": url, "error": error_msg})
                    break

            if not success:
                print(f"✗ All {MAX_RETRIES} retries failed for: {filename}")

        return results

    except requests.exceptions.Timeout:
        error_msg = f"Timeout while accessing: {url}"
        print(f"✗ {error_msg}")
        return [{"status": "timeout", "url": url, "error": error_msg}]
    except requests.exceptions.ConnectionError:
        error_msg = f"Connection error while accessing: {url}"
        print(f"✗ {error_msg}")
        return [{"status": "connection_error", "url": url, "error": error_msg}]
    except requests.exceptions.HTTPError as e:
        error_msg = f"HTTP error {e.response.status_code} while accessing: {url}"
        print(f"✗ {error_msg}")
        return [{"status": "http_error", "url": url, "error": error_msg}]
    except requests.exceptions.RequestException as e:
        error_msg = f"Request error while accessing {url}: {e}"
        print(f"✗ {error_msg}")
        return [{"status": "request_error", "url": url, "error": error_msg}]
    except Exception as e:
        error_msg = f"Unexpected error while processing {url}: {e}"
        print(f"✗ {error_msg}")
        return [{"status": "unexpected_error", "url": url, "error": error_msg}]

def download_pdfs_parallel(all_acts_links, max_workers=5):
    """
    Download PDFs from multiple URLs in parallel
    """
    print(f"Starting parallel download of {len(all_acts_links)} URLs with {max_workers} workers...")

    results = {
        "success": [],
        "errors": [],
        "no_pdf_found": [],
        "total_processed": 0
    }

    start_time = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(download_pdf_from_url, url, i, len(all_acts_links)): url 
                   for i, url in enumerate(all_acts_links)}

        for future in as_completed(futures):
            self_results = future.result()
            results["total_processed"] += 1
            for r in self_results:
                if r["status"] == "success":
                    results["success"].append(r)
                elif r["status"] == "no_pdf_found":
                    results["no_pdf_found"].append(r)
                else:
                    results["errors"].append(r)

    end_time = time.time()

    # Print summary
    print("\n" + "="*50)
    print("DOWNLOAD SUMMARY")
    print("="*50)
    print(f"Total URLs processed: {results['total_processed']}")
    print(f"Successful downloads: {len(results['success'])}")
    print(f"URLs with no PDF found: {len(results['no_pdf_found'])}")
    print(f"Errors encountered: {len(results['errors'])}")
    print(f"Total time taken: {end_time - start_time:.2f} seconds")

    if results["errors"]:
        print("\nERROR DETAILS:")
        print("-" * 30)
        for err in results["errors"]:
            print(f"URL: {err['url']}")
            print(f"Error: {err.get('error', 'N/A')}")
            print(f"Status: {err['status']}")
            print("-" * 30)

    return results


In [5]:
results = download_pdfs_parallel(all_acts_links, max_workers=15)

Starting parallel download of 10861 URLs with 15 workers...
Processing link 1/10861: https://www.indiacode.nic.in/handle/123456789/3111?view_type=search&col=123456789/1362
Processing link 2/10861: https://www.indiacode.nic.in/handle/123456789/3113?view_type=search&col=123456789/1362
Processing link 3/10861: https://www.indiacode.nic.in/handle/123456789/3112?view_type=search&col=123456789/1362
Processing link 4/10861: https://www.indiacode.nic.in/handle/123456789/4315?view_type=search&col=123456789/1362
Processing link 5/10861: https://www.indiacode.nic.in/handle/123456789/12850?view_type=search&col=123456789/1362
Processing link 6/10861: https://www.indiacode.nic.in/handle/123456789/12851?view_type=search&col=123456789/1362
Processing link 7/10861: https://www.indiacode.nic.in/handle/123456789/12852?view_type=search&col=123456789/1362
Processing link 8/10861: https://www.indiacode.nic.in/handle/123456789/12853?view_type=search&col=123456789/1362
Processing link 9/10861: https://www.ind

In [9]:
import os

# Path to the folder with your PDF files
folder_path = out_folder

# Loop through all files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".pdf") and "&file=" in filename:
        new_filename = filename.replace("&file=", "")
        old_file = os.path.join(folder_path, filename)
        new_file = os.path.join(folder_path, new_filename)
        try:
            os.rename(old_file, new_file)
        except Exception as e:
            print(f"Error renaming {filename}: {e}")
            continue
        print(f"Renamed: {filename} -> {new_filename}")


Error renaming &file=1%29_agricultural_tenancy_act_and_rules_1964-not_ocr.pdf: [WinError 183] Cannot create a file when that file already exists: './acts_pdf\\&file=1%29_agricultural_tenancy_act_and_rules_1964-not_ocr.pdf' -> './acts_pdf\\1%29_agricultural_tenancy_act_and_rules_1964-not_ocr.pdf'
Renamed: &file=1++June+20+1989.pdf -> 1++June+20+1989.pdf
Renamed: &file=1+10+Feb+2006+SEZ+Rules%2C+2006-+Principle+Rules+GSR+No.+54+E+dated+10.02.2006.pdf -> 1+10+Feb+2006+SEZ+Rules%2C+2006-+Principle+Rules+GSR+No.+54+E+dated+10.02.2006.pdf
Renamed: &file=1+January+1+2003.pdf -> 1+January+1+2003.pdf
Renamed: &file=1+march+2017.pdf -> 1+march+2017.pdf
Renamed: &file=1+of+2001.pdf -> 1+of+2001.pdf
Renamed: &file=1+of+2002+-+Corrigendum.pdf -> 1+of+2002+-+Corrigendum.pdf
Renamed: &file=1+of+2002.pdf -> 1+of+2002.pdf
Renamed: &file=1+of+2003.pdf -> 1+of+2003.pdf
Renamed: &file=1+of+2005.pdf -> 1+of+2005.pdf
Renamed: &file=1+of+2006.pdf -> 1+of+2006.pdf
Renamed: &file=1+of+2007.pdf -> 1+of+2007.pdf

In [10]:
import os 

len(os.listdir(folder_path))

42828